# ClinGuard-DiLoCo: Kaggle session 2 (gate G3: zero-shot, first fine-tune, resume check)

Runs three jobs through `scripts/run_job.py`. Every round is uploaded to the private HF repo with a lease,
so a crash or timeout loses at most one round: rerun the notebook and each job continues where it stopped.

| GPU | Jobs, in order | Estimate |
|---|---|---|
| 0 | zero-shot reports on development data (3 sites); then the resume check (two short 3-site runs) | ~40 min |
| 1 | site A fine-tuned alone for 1,000 optimizer steps, then development reports for all 3 sites | ~80 min |

**Settings** (right-hand panel): GPU **T4 x2**, Internet **on**, input dataset **raddar/chest-xrays-indiana-university**,
secret **HF_TOKEN** attached. Keep the notebook **private**.

**How to run:** *Save Version* -> **Save & Run All (Commit)**. It keeps running with the browser closed
(up to 12 h). Then open the version's *Logs* / *Output*: the last cell prints a summary.

In [ ]:
HF_REPO = "tasbid66/clinguard-runs"
GIT_REF = "main"     # a resumed run refuses a different commit; to resume after main moved, set the commit printed in its log

In [ ]:
import glob, os, subprocess, sys, time, json
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!rm -rf /kaggle/working/ClinGuard && git clone -q https://github.com/tasbidrahman10/ClinGuard.git /kaggle/working/ClinGuard
%cd /kaggle/working/ClinGuard
!git checkout -q {GIT_REF} && git log --oneline -1
!pip install -q "transformers==5.23.0" "peft==0.22.0" "accelerate==1.16.0" bitsandbytes pyyaml
!pip install -q --no-deps -e .
src = os.path.dirname(glob.glob("/kaggle/input/**/indiana_reports.csv", recursive=True)[0])
os.makedirs("data/raw", exist_ok=True)
if not os.path.exists("data/raw/iu_xray"):
    os.symlink(src, "data/raw/iu_xray")
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

Download the 3B model once, so the two jobs do not race for it.

In [ ]:
from huggingface_hub import snapshot_download
sys.path.insert(0, "src")
from clinguard.model import load
snapshot_download(load.MODEL_ID, revision=load.MODEL_REVISION)
print("model cached")

## Launch the jobs (one per GPU, in the background)

In [ ]:
os.makedirs("/kaggle/working/logs", exist_ok=True)
RUN = f"python scripts/run_job.py --hf-repo {HF_REPO} --work-dir /kaggle/working/runs --config configs/experiment"
gpu0 = " && ".join([f"{RUN}/s0_zero_shot.yaml",
                    f"{RUN}/g3_resume_straight.yaml",
                    f"{RUN}/g3_resume_split.yaml --stop-after-round 1",
                    f"{RUN}/g3_resume_split.yaml"])
gpu1 = f"{RUN}/s1_single_site_A.yaml"
procs = {}
for gpu, cmd in ((0, gpu0), (1, gpu1)):
    log = open(f"/kaggle/working/logs/gpu{gpu}.log", "w")
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONUNBUFFERED": "1"}
    procs[gpu] = subprocess.Popen(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT, env=env)
    print("GPU", gpu, "started:", cmd.replace(" && ", "
   then "))

## Wait, printing progress every 5 minutes

In [ ]:
def tail(path, n=3):
    lines = [l for l in open(path, errors="replace").read().splitlines() if l.strip() and "Warning" not in l]
    return lines[-n:]
start = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    print(f"--- {int(time.time() - start) // 60} min")
    for gpu in procs:
        state = "running" if procs[gpu].poll() is None else f"exit {procs[gpu].returncode}"
        print(f"GPU {gpu} ({state}):", *tail(f"/kaggle/working/logs/gpu{gpu}.log"), sep="
   ")
print({gpu: p.returncode for gpu, p in procs.items()})

## Summary

In [ ]:
import importlib.util, torch
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
spec = importlib.util.spec_from_file_location("rj", "scripts/run_job.py"); rj = importlib.util.module_from_spec(spec); spec.loader.exec_module(rj)
ids = {n: rj.run_id_for(rj.load_config(f"configs/experiment/{n}.yaml"))
       for n in ("s0_zero_shot", "s1_single_site_A", "g3_resume_straight", "g3_resume_split")}
def fetch(run_id, name):
    try:
        return hf_hub_download(HF_REPO, f"runs/{run_id}/{name}", force_download=True)
    except Exception as e:
        print("missing:", run_id, name, type(e).__name__); return None
for n, rid in ids.items():
    lease = fetch(rid, "LEASE.json")
    print(f"{n:20s} {rid}  status: {json.load(open(lease))['status'] if lease else '-'}")
r = fetch(ids["s1_single_site_A"], "rounds.jsonl")
if r:
    for row in map(json.loads, open(r)):
        c = row["cost"]["A"]
        print(f"  s1 round {row['round']}: train loss {row['train_loss']['A']:.4f}  |d| {row['delta_norm']['A']:.4f}  "
              f"{c['train_sec'] / 100:.2f} s/step  unique studies {c['unique']}")
a, b = (fetch(ids[k], "phi_final.safetensors") for k in ("g3_resume_straight", "g3_resume_split"))
if a and b:
    pa, pb = load_file(a)["phi"], load_file(b)["phi"]
    print("resume check (A10): bit-identical:", torch.equal(pa, pb), "| max |diff|:", (pa - pb).abs().max().item(),
          "| relative:", ((pa - pb).norm() / pa.norm()).item(), "| |phi|:", pa.norm().item())

**Send Tasbid / Claude:** the output of the last two cells. Generated reports stay in the HF repo; they are scored locally in the evaluation environment.